#### Part 1: Basis approach using decision trees

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from tensorflow.keras.datasets import fashion_mnist

In [ ]:
# Load the data

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

In [ ]:
# Preprocess the data: decision trees expect 2D input, so we need to flatten the images
x_train_flat = x_train.reshape(x_train.shape[0], -1) 
x_test_flat = x_test.reshape(x_test.shape[0], -1)

In [10]:
# Decision Tree Classifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(x_train_flat, y_train)
 
y_pred = clf.predict(x_test_flat)
tree_train_acc = clf.score(x_train_flat, y_train)
tree_test_acc = accuracy_score(y_test, y_pred)
 
print("\n=== Baseline Decision Tree (no depth limit) ===")
print(f"Train accuracy: {tree_train_acc:.4f}")
print(f"Test accuracy:  {tree_test_acc:.4f}")
print(f"Tree depth reached: {clf.get_depth()}")
print(f"Number of leaves: {clf.get_n_leaves()}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


=== Baseline Decision Tree (no depth limit) ===
Train accuracy: 1.0000
Test accuracy:  0.7890
Tree depth reached: 49
Number of leaves: 4910

Classification report:
              precision    recall  f1-score   support

 T-shirt/top       0.76      0.73      0.75      1000
     Trouser       0.96      0.95      0.95      1000
    Pullover       0.63      0.65      0.64      1000
       Dress       0.82      0.78      0.80      1000
        Coat       0.64      0.63      0.64      1000
      Sandal       0.90      0.89      0.89      1000
       Shirt       0.52      0.55      0.54      1000
     Sneaker       0.87      0.88      0.88      1000
         Bag       0.91      0.91      0.91      1000
  Ankle boot       0.90      0.91      0.91      1000

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000



In [11]:
# Random Forest Ensemble
rf = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",   # random feature subset per split -> decorrelates trees
    n_jobs=-1,
    oob_score=True,        # free validation estimate from out-of-bag samples
    random_state=42,
)
rf.fit(x_train_flat, y_train)
 
y_pred_rf = rf.predict(x_test_flat)
rf_train_acc = rf.score(x_train_flat, y_train)
rf_test_acc = accuracy_score(y_test, y_pred_rf)
 
print("=== Comparison: Single Tree vs Random Forest ===")
print(f"{'Model':<20}{'Train Acc':>12}{'Test Acc':>12}{'Gap':>10}")
print(f"{'Decision Tree':<20}{tree_train_acc:>12.4f}{tree_test_acc:>12.4f}"
      f"{tree_train_acc - tree_test_acc:>10.4f}")
print(f"{'Random Forest':<20}{rf_train_acc:>12.4f}{rf_test_acc:>12.4f}"
      f"{rf_train_acc - rf_test_acc:>10.4f}")
print(f"\nRandom Forest OOB score (train-set-derived generalization estimate): "
      f"{rf.oob_score_:.4f}")
 
print("\nClassification report (Random Forest):")
print(classification_report(y_test, y_pred_rf, target_names=CLASS_NAMES))


=== Comparison: Single Tree vs Random Forest ===
Model                  Train Acc    Test Acc       Gap
Decision Tree             1.0000      0.7890    0.2110
Random Forest             1.0000      0.8760    0.1240

Random Forest OOB score (train-set-derived generalization estimate): 0.8806

Classification report (Random Forest):
              precision    recall  f1-score   support

 T-shirt/top       0.82      0.86      0.84      1000
     Trouser       0.99      0.96      0.98      1000
    Pullover       0.77      0.81      0.79      1000
       Dress       0.88      0.90      0.89      1000
        Coat       0.77      0.82      0.79      1000
      Sandal       0.98      0.96      0.97      1000
       Shirt       0.71      0.58      0.64      1000
     Sneaker       0.93      0.95      0.94      1000
         Bag       0.96      0.97      0.97      1000
  Ankle boot       0.95      0.94      0.95      1000

    accuracy                           0.88     10000
   macro avg       